In [ ]:
!pip install contractions langdetect nltk beautifulsoup4 lxml --quiet

In [ ]:
import pandas as pd
import numpy as np
import re
import html
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from langdetect import detect, LangDetectException, DetectorFactory
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

DetectorFactory.seed = 42

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
raw_df = pd.read_csv("taylor_swift_reddit_rss_raw_dataset.csv")

print("Raw dataset shape:", raw_df.shape)
print("Columns:")
print(raw_df.columns.tolist())

raw_df.head()

In [ ]:
def normalise_raw_text(text):
    """
    Convert RSS/HTML text into readable plain text.
    This removes HTML entities and extra spacing.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    text = text.replace("\u200b", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


df = raw_df.copy()

df["text_readable"] = df["text_for_analysis"].apply(normalise_raw_text)

df[["text_for_analysis", "text_readable"]].head(10)

In [ ]:
initial_rows = len(df)

# Remove RSS post artefacts such as:
# "submitted by /u/username [link] [comments]"
rss_artifact_pattern = r"^submitted by\s+/u/\S+\s+\[link\]\s+\[comments\]$"

df["is_rss_artifact"] = df["text_readable"].str.contains(
    rss_artifact_pattern,
    case=False,
    regex=True,
    na=False
)

# Remove deleted, removed, empty and meaningless rows
df["is_deleted_or_empty"] = df["text_readable"].str.lower().isin(
    ["", "[deleted]", "[removed]", "deleted", "removed", "nan"]
)

before_filter = len(df)

df = df[
    (df["is_rss_artifact"] == False) &
    (df["is_deleted_or_empty"] == False)
].copy()

after_artifact_filter = len(df)

# Remove duplicate comments from the same post
before_duplicates = len(df)

df = df.drop_duplicates(subset=["post_link", "text_readable"]).copy()

after_duplicates = len(df)

print("Initial rows:", initial_rows)
print("Rows after removing RSS artefacts/deleted/empty rows:", after_artifact_filter)
print("Rows after removing duplicates:", after_duplicates)
print("Rows removed in total:", initial_rows - after_duplicates)

In [ ]:
df["raw_char_count"] = df["text_readable"].apply(len)
df["raw_word_count"] = df["text_readable"].apply(lambda x: len(str(x).split()))

before_short_filter = len(df)

# Keep comments with at least 10 characters and at least 3 words.
# This removes very weak comments like "ok", "same", "lol".
df = df[
    (df["raw_char_count"] >= 10) &
    (df["raw_word_count"] >= 3)
].copy()

after_short_filter = len(df)

print("Rows before short-text filter:", before_short_filter)
print("Rows after short-text filter:", after_short_filter)
print("Very short rows removed:", before_short_filter - after_short_filter)

df[["text_readable", "raw_char_count", "raw_word_count"]].head()

In [ ]:
def detect_language_safely(text):
    """
    Detect the language of a comment.
    If detection fails, return 'unknown'.
    """
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"


df["lang"] = df["text_readable"].apply(detect_language_safely)

print("Language distribution:")
print(df["lang"].value_counts().head(10))

before_language_filter = len(df)

df = df[df["lang"] == "en"].copy()

after_language_filter = len(df)

print("\nRows before language filter:", before_language_filter)
print("Rows after keeping English only:", after_language_filter)
print("Rows removed by language filter:", before_language_filter - after_language_filter)

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Add project-specific stopwords.
# These words are common in the dataset but may not be useful for interpretation.
custom_stopwords = {
    "taylor", "swift", "reddit", "comment", "comments",
    "link", "submitted", "amp", "https", "http", "www",
    "like", "just", "really", "would", "could", "also"
}

all_stopwords = stop_words.union(custom_stopwords)


def clean_for_text_mining(text):
    """
    Clean Reddit comment text for text mining.
    """
    text = str(text)

    # Expand contractions: don't -> do not
    text = contractions.fix(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove Reddit usernames and subreddit mentions
    text = re.sub(r"u/\w+", " ", text)
    text = re.sub(r"r/\w+", " ", text)

    # Remove punctuation, numbers and symbols
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenise using simple split
    words = text.split()

    # Remove stopwords, very short words, and lemmatise
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in all_stopwords and len(word) > 2
    ]

    return " ".join(words)


df["cleaned_text"] = df["text_readable"].apply(clean_for_text_mining)

df[["text_readable", "cleaned_text"]].head(10)

In [ ]:
before_empty_cleaned = len(df)

df = df[df["cleaned_text"].str.len() > 0].copy()

after_empty_cleaned = len(df)

print("Rows before removing empty cleaned text:", before_empty_cleaned)
print("Rows after removing empty cleaned text:", after_empty_cleaned)
print("Rows removed:", before_empty_cleaned - after_empty_cleaned)

In [ ]:
df["comment_datetime"] = pd.to_datetime(df["comment_datetime"], errors="coerce")
df["comment_date"] = df["comment_datetime"].dt.date
df["comment_month"] = df["comment_datetime"].dt.to_period("M").astype(str)
df["comment_year"] = df["comment_datetime"].dt.year

print("Earliest comment:", df["comment_datetime"].min())
print("Latest comment:", df["comment_datetime"].max())

df[["comment_datetime", "comment_date", "comment_month", "comment_year"]].head()

In [ ]:
privacy_drop_columns = [
    "comment_author_raw"
]

for col in privacy_drop_columns:
    if col in df.columns:
        df = df.drop(columns=[col])

print("Columns after privacy cleaning:")
print(df.columns.tolist())

In [ ]:
summary = {
    "Final number of comments": len(df),
    "Number of unique posts": df["post_link"].nunique(),
    "Number of unique anonymised authors": df["comment_author_id"].nunique(),
    "Earliest comment date": df["comment_datetime"].min(),
    "Latest comment date": df["comment_datetime"].max(),
    "Average raw comment length": round(df["raw_char_count"].mean(), 2),
    "Median raw comment length": round(df["raw_char_count"].median(), 2),
    "Average raw word count": round(df["raw_word_count"].mean(), 2),
    "Median raw word count": round(df["raw_word_count"].median(), 2),
    "Shortest comment length": df["raw_char_count"].min(),
    "Longest comment length": df["raw_char_count"].max()
}

summary_df = pd.DataFrame(summary.items(), columns=["Metric", "Value"])

summary_df

In [ ]:
sample_for_report = df[
    ["post_title", "text_readable", "cleaned_text", "raw_char_count", "raw_word_count", "comment_datetime"]
].head(10)

sample_for_report

In [ ]:
cleaned_filename = "taylor_swift_reddit_cleaned_dataset.csv"
summary_filename = "taylor_swift_reddit_dataset_summary.csv"

df.to_csv(cleaned_filename, index=False)
summary_df.to_csv(summary_filename, index=False)

print("Saved cleaned dataset as:", cleaned_filename)
print("Saved summary table as:", summary_filename)